In [1]:
import os 
from dotenv import load_dotenv

import re
import random
import sqlite3
import pandas as pd
import sqlalchemy
from sqlalchemy.exc import OperationalError

from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.callbacks import BaseCallbackHandler
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_classic.chains.sql_database.query import create_sql_query_chain

pd.set_option('display.max_columns', None)

In [2]:
# ACTIVATE GOOGLE GEMINI API

# GET API KEY FROM .env
load_dotenv()
GOOGLE_API_KEY = os.getenv('GEMINI_API_KEY')

# DEFINE GEMINI MODEL 
MODEL_NAME = "models/gemma-3-27b-it"
llm = ChatGoogleGenerativeAI(model = MODEL_NAME,  # USE GEMMA MODEL FOR CHEAPER MODEL
                             temperature = 0      # Temperature 0 agar jawabannya deterministik dan akurat secara data
                             ) 

print(f'Connected to Gemini API : {GOOGLE_API_KEY[:5]}***')

Connected to Gemini API : AIzaS***


In [3]:
# CONNECT TO DATABASE SQLITE3

DATABASE_NAME = 'financial.db'
TABLE_NAME = 'financials_data'

# CREATE CUSTOM TABLE DESCRIPTION
custom_table_description = {"financials_data": """Table containing financial metrics of companies. 
                            1. "Symbol" TEXT, 
                            2. "Name" TEXT, 
                            3. "Sector" TEXT, 
                            4. "Price" REAL --> current price, 
                            5. "Price_Earnings" REAL, 
                            6. "Dividend_Yield" REAL --> stored as actual percentage number (e.g., 3.5 means 3.5%, do NOT convert to 0.035),
                            7. "Earnings_Share" REAL, 
                            8. "52_Week_Low" REAL --> the lowest price in the last 1 year, 
                            9. "52_Week_High" REAL --> the highest price in the last 1 year, 
                            10. "Market_Cap" REAL, 
                            11. "EBITDA" REAL  --> Earnings Before Interest, Taxes, Depreciation, and Amortization., 
                            12. "Price_Sales" REAL, 
                            13. "Price_Book" REAL, 
                            14. "SEC_Filings" TEXT"""}

# CONNECT SQLITE DATABASE
db = SQLDatabase.from_uri(database_uri = f"sqlite:///{DATABASE_NAME}", sample_rows_in_table_info = 0, custom_table_info = custom_table_description)

In [4]:
# PROMPT

PROMPT = PromptTemplate.from_template("""You are a SQLite SQL generator. 
            Database: financials_data
            
            You must comply with all of the following rules:
            - Use valid SQLite syntax
            - Use single quotes for string
            - If a column name starts with a number (e.g., 52 Week High) or has spaces, you MUST wrap it ONLY in square brackets. Example: [52_Week_High]
            - Return ONLY SQL query , NO EXPLANATION, NO PYTHON CODE.
            Here is the table schema:{table_info}
            don't limit your query {top_k} results unless otherwise specified.
            Question: {input}
            SQLQuery:""")

In [5]:
# TEXT-TO-SQL

query_chain = create_sql_query_chain(llm = llm, db = db, prompt = PROMPT)
query_chain

RunnableAssign(mapper={
  input: RunnableLambda(...),
  table_info: RunnableLambda(...)
})
| RunnableLambda(lambda x: {k: v for k, v in x.items() if k not in ('question', 'table_names_to_use')})
| PromptTemplate(input_variables=['input', 'table_info'], input_types={}, partial_variables={'top_k': '5'}, template="You are a SQLite SQL generator. \n            Database: financials_data\n            \n            You must comply with all of the following rules:\n            - Use valid SQLite syntax\n            - Use single quotes for string\n            - If a column name starts with a number (e.g., 52 Week High) or has spaces, you MUST wrap it ONLY in square brackets. Example: [52_Week_High]\n            - Return ONLY SQL query , NO EXPLANATION, NO PYTHON CODE.\n            Here is the table schema:{table_info}\n            don't limit your query {top_k} results unless otherwise specified.\n            Question: {input}\n            SQLQuery:")
| RunnableBinding(bound=ChatGoogleGenerativ

In [6]:
# FUNCTION TO CLEAN SQL OUTPUT (REMOVE MARKDOWN CHARACTER)
def clean_sql_output(sql: str):
    
    # REMOVE BACKTICKS (MARKDOWN)
    sql = re.sub(r"```.*?\n", "", sql)
    sql = sql.replace("```", "")

    return sql.strip()

In [7]:
# FUNCTION TO DISPLAY THE OUTPUT OF QUERY

# CONNECT SQLITE3 DATABASE WITH SQLALCHEMY
alchemy_db = sqlalchemy.create_engine(url = "sqlite:///financial.db")



# DISPLAY IN DATAFRAME FORM
def display_output(sql_query : str):

    # SEARCH QUERY IN DATABASE
    df = pd.read_sql(sql = sql_query, con = alchemy_db)
    
    return df

In [8]:
# CALLBACK TO TRACK TOKEN CONSUMPTION

class TokenCounter(BaseCallbackHandler):

    # SIGNATURE
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.total_tokens = 0
        self.calls = 0

    # AT THE END ON EACH ITERATION
    def on_llm_end(self, response, **kwargs):

        # NUMBER OF ITERATION
        self.calls += 1
        
        # METADATA
        usage = response.generations[0][0].message.usage_metadata
        
        # GET TOKEN USAGE EACH ITERATION
        input_tokens = usage.get("input_tokens", 0)
        output_tokens = usage.get("output_tokens", 0)
        total_tokens = usage.get("total_tokens", 0)
        
        # SUM TOKEN USAGE EACH ITERATION
        self.total_input += input_tokens
        self.total_output += output_tokens
        self.total_tokens += total_tokens
        
counter = TokenCounter()

In [9]:
# FULL PIPELINE

def pipeline(question, model = query_chain, display_token = False):

    # DEFINE CALLBACKS (TOKEN USAGE TRACKER)
    counter = TokenCounter()

    # GENERATE SQL (TEXT-TO-SQL)
    sql_query = model.invoke({'question' : question}, config = {'callbacks' : [counter]})

    # CLEAN LLM OUTPUT
    sql_query = clean_sql_output(sql_query)

    # DISPLAY OUTPUT AS DATAFRAME
    df_result = display_output(sql_query)

    # DISPLAY TOKEN USAGE?
    if display_token:
        print("Total Iterations:", counter.calls)
        print("Total Input Tokens:", counter.total_input)
        print("Total Output Tokens:", counter.total_output)
        print("Grand Total Tokens:", counter.total_tokens)

    return {'query' : sql_query, 'answer' : df_result, 'token_usage' : counter}


In [10]:
# TEST
response = query_chain.invoke(input = {'question' : "what is the price of apple stock?"}, config = {'callbacks' : [counter]})

# DISPLAY OUTPUT
display(response)

# TOKEN USAGE
print("Total Iterations:", counter.calls)
print("Total Input Tokens:", counter.total_input)
print("Total Output Tokens:", counter.total_output)
print("Grand Total Tokens:", counter.total_tokens)

"```sqlite\nSELECT Price FROM financials_data WHERE Symbol = 'AAPL'\n```"

Total Iterations: 1
Total Input Tokens: 397
Total Output Tokens: 0
Grand Total Tokens: 397


In [10]:
# TEST
response = query_chain.invoke(input = {'question' : "What are the top 5 tech companies by market cap?"}, config = {'callbacks' : [counter]})

# DISPLAY OUTPUT
display(response)

# TOKEN USAGE
print("Total Iterations:", counter.calls)
print("Total Input Tokens:", counter.total_input)
print("Total Output Tokens:", counter.total_output)
print("Grand Total Tokens:", counter.total_tokens)

"```sqlite\nSELECT Symbol, Name, Market_Cap FROM financials_data WHERE Sector = 'Technology' ORDER BY Market_Cap DESC LIMIT 5\n```"

Total Iterations: 1
Total Input Tokens: 401
Total Output Tokens: 0
Grand Total Tokens: 401


In [12]:
# TEST CASE #1

response = pipeline(question = "Tampilkan simbol dan nama perusahaan yang harga sahamnya saat ini lebih tinggi dari harga tertingginya dalam setahun terakhir.", display_token = True)

# DISPLAY OUTPUT
display(response)

# DISPLAY OUTPUT
display(response['query'])
display(response['answer'])

Total Iterations: 1
Total Input Tokens: 414
Total Output Tokens: 0
Grand Total Tokens: 414


{'query': 'SELECT Symbol, Name FROM financials_data WHERE Price > [52_Week_High]',
 'answer': Empty DataFrame
 Columns: [Symbol, Name]
 Index: [],
 'token_usage': <__main__.TokenCounter at 0x1aedf96dd30>}

'SELECT Symbol, Name FROM financials_data WHERE Price > [52_Week_High]'

,Symbol,Name


In [13]:
# TEST CASE #2 

QUESTION = "Tampilkan nama dan sektor dari perusahaan yang dianggap 'undervalued' atau salah harga. Kriterianya: rasio harga terhadap laba di bawah 15, " \
            "rasio harga terhadap nilai buku di bawah 1.5, dan perusahaannya masih mencetak laba per lembar saham yang positif. " \
            "Urutkan hasilnya berdasarkan persentase laba per lembar saham jika dibandingkan dengan harga sahamnya saat ini, dari yang rasionya paling besar."

response = pipeline(question = QUESTION, display_token = True)

# DISPLAY OUTPUT
display(response['query'])
display(response['answer'])

Total Iterations: 1
Total Input Tokens: 484
Total Output Tokens: 0
Grand Total Tokens: 484


'SELECT Name, Sector FROM financials_data WHERE Price_Earnings < 15 AND Price_Book < 1.5 AND Earnings_Share > 0 ORDER BY (Earnings_Share / Price) DESC'

,Name,Sector
0,"Allergan, Plc",Health Care
1,Ford Motor,Consumer Discretionary
2,General Motors,Consumer Discretionary
3,Signet Jewelers,Consumer Discretionary
4,Lincoln National,Financials
5,SCANA Corp,Utilities
6,Prudential Financial,Financials
7,Unum Group,Financials
8,Discovery Communications-C,Consumer Discretionary
9,Navient,Financials


In [19]:
# TEST CASE #3

QUESTION = "what stock is the most recommended to invest in? show me the company top 10"

response = pipeline(question = QUESTION, display_token = True)

# DISPLAY OUTPUT
display(response['query'])
display(response['answer'])

Total Iterations: 1
Total Input Tokens: 407
Total Output Tokens: 0
Grand Total Tokens: 407


'SELECT Symbol, Name FROM financials_data ORDER BY (Price_Earnings * Dividend_Yield * Earnings_Share) DESC LIMIT 10'

,Symbol,Name
0,APA,Apache Corporation
1,BLK,BlackRock
2,CME,CME Group Inc.
3,RE,Everest Re Group Ltd.
4,APD,Air Products & Chemicals Inc
5,NSC,Norfolk Southern Corp.
6,AIZ,Assurant Inc
7,BA,Boeing Company
8,AGN,"Allergan, Plc"
9,NEE,NextEra Energy
